In [1]:
import pdftotext
from tabula.io import read_pdf       # wrapper of tabula, table extraction from pdf

import numpy 
import pandas as pd

import os
import glob

In [2]:
path="/home/aman/Desktop/HaystackAnalytics/Task1/genomeanalysisfinalreport/INPUT_PDF/" 
output_path = '/home/aman/Desktop/HaystackAnalytics/Task1/genomeanalysisfinalreport/OUTPUT/'
list_filenames = glob.glob(path+"*.pdf")
list_folder_names = glob.glob("*.pdf")
#print(list_folder_names)

#print(list_filenames)
for path_file in list_filenames:

    
    outputfile_name = path_file.split("/")[-1]
    output_folder_name = path_file.split("/")[-1]
    output_folder_name = output_folder_name.split(".")[0]+"/"
    #print(output_folder_name)

    output_data = output_path+output_folder_name+outputfile_name
    #print(output_data)

    if os.path.exists(output_path+output_folder_name):
        continue
    else:
        os.makedirs(output_path+output_folder_name)

    with open(path_file, "rb") as f:
        pdf = pdftotext.PDF(f)


        
        # Extracting metadata
        #print(pdf[0])
        page_1_text= pdf[0]
        page_1_text= page_1_text.split('\n')
        #print(page_1_text)

        def get_data(page_1_text):
            __dict = {}

            for line in page_1_text:
                if ':' not in line:
                    continue
                key, value = line.split(':')
                __dict[key.strip()] = value.strip()
            return __dict

        page_data = get_data(page_1_text)
        
        with open(output_data.replace('.pdf', '_metadata.csv'), 'w+') as output:
            for key in page_data.keys():
                output.write("%s,%s\n"%(key,page_data[key]))

          

        # Extractiong sample summary
        page_2_text = pdf[1]
        page_2_text = page_2_text.split('\n')

        def get_data(page_2_text):
            __dict = {}
    
            for line in page_2_text:
                if ':' not in line:
                    continue
                key, value = line.split(':')
                __dict[key.strip()] = value.strip()
            return __dict

        page_data = get_data(page_2_text)

        with open(output_data.replace('.pdf', '_sample_summary.csv'), 'w') as f:
            for key in page_data.keys():
                f.write("%s,%s\n"%(key,page_data[key]))

        
        # extracting dst_table

        page_3_text = pdf[2]
        page_3_text = page_3_text.split('\n')


        # extracting dst_table

        def get_data(page_3_text):
            __dict = {}
            for line in page_3_text:
                if ')' in line:
                    key, value = line.split(')')
                    __dict[key.strip()] = value.strip()
                elif 'Clinical Recommendations' in line:
                    break
            return __dict

        page_data = get_data(page_3_text)
        
        with open(output_data.replace('.pdf', '_dst_table.csv'), 'w') as f:
            for key in page_data.keys():
                f.write("%s,%s\n"%(key,page_data[key]))

        
    
        # extracting clinical summary
        clinical_summary = open(path_file.replace('.pdf','.txt'),'wb')
        text = pdf[2].encode("utf8")

        #print(text)

        clinical_summary.write(text)
        clinical_summary.write(bytes((12,)))  # write page delimiter (form feed 0x0C)   # decimal to hexadecimal
        clinical_summary.close()

        
        text = open(path_file.replace('pdf','txt'), "rt")
        readtext =text.readlines()
        
        del readtext[0]
        del readtext[2::]

        #print(readtext)

        new_file = open(path_file.replace('.pdf','.txt'),"w+")

        for line in readtext:
            line = line.replace("-\n","")
            line = line.replace("\n",",").strip()
            line = line.replace("    ",",").strip()

            #print(line)            
            new_file.write(line)
                    
        new_file.close()
            

        dataframe = pd.read_csv(path_file.replace('.pdf','.txt'))

        dataframe = dataframe.loc[:,~dataframe.columns.str.match("Unnamed")]
        dataframe.T.to_csv(output_data.replace('.pdf','_clinical_summary.csv'),header=False)

        dataframe = pd.read_csv(output_data.replace('.pdf','_clinical_summary.csv'))

        x=dataframe.columns
        a=(str(dataframe.columns).replace("Index(['", "")).replace("'], dtype='object')", "")

        col=a
        row=dataframe[a][0]

        for i in range(int(len(dataframe)/2)):
            col=col+','+dataframe[a][2*i+1]
            row=row+','+dataframe[a][2*i+2]
    
        new_csv=open(output_data.replace('.pdf','_clinical_summary.csv'),'w')
        new_csv.write(col+'\n'+row)
        new_csv.close()

        # Extracting mutation table

        df = read_pdf(path_file, pages=5, stream=True)
        if df == []:
            sample_data = open(output_data.replace('.pdf','_mutation_table.csv'),'w')
            sample_data.write("No Mutations were Detected")
            sample_data.close()
            #print("No Mutations were Detected!")
        else:
            col=len(df[0].columns)
            x='SN'         # titles adding to x
            for i in range(col):
                x=x+','+(df[0].columns)[i]
            value=(len(df[0][(df[0].columns)[0]]))    # total rows present
            w=''                              # row value
            for i in range(col):
                w=w+str(i)
                for j in range(value):
                    w=w+','+str(numpy.array(df[0][((df[0].columns)[j])])[i])
                w=w+"\n"
            sample_data=open(output_data.replace('.pdf','_mutation_table.csv'),'w')
            sample_data.write(x+'\n'+w)
            sample_data.close()



        # using pandas to edit csv files 
        # page 1 csv file    # metadata

        df = pd.read_csv(output_data.replace('.pdf', '_metadata.csv'))
        df = df.iloc[:-1]
        df.T.to_csv(output_data.replace('.pdf', '_metadata.csv'),header=False)

        # page 2 data     # sample_summary

        df = pd.read_csv(output_data.replace('.pdf', '_sample_summary.csv'))
        df = df.iloc[:-1]
        df.T.to_csv(output_data.replace('.pdf', '_sample_summary.csv'),header=False)

        # page 3 csv     # dst_table

        df = pd.read_csv(output_data.replace('.pdf', '_dst_table.csv'))        
        #df = df.iloc[:-1]
        df.T.to_csv(output_data.replace('.pdf', '_dst_table.csv'),header=False)

        df = pd.read_csv(output_data.replace('.pdf', '_dst_table.csv'))
        df = df.add_suffix(")")
        df.to_csv(output_data.replace('.pdf', '_dst_table.csv'),index=None)


        # page 4 csv           # mutation_table

        df = pd.read_csv(output_data.replace('.pdf','_mutation_table.csv'))
        #print(df.columns[0])
        if df.columns[0] != "SN":
            continue
        else:
            df.pop('SN')
            df = df.fillna("")
            df.to_csv(output_data.replace('.pdf','_mutation_table.csv'), index=False)


        
        




In [ ]:
path="/home/aman/Desktop/HaystackAnalytics/Task1/genomeanalysisfinalreport/INPUT_PDF/" 
output_path = '/home/aman/Desktop/HaystackAnalytics/Task1/genomeanalysisfinalreport/OUTPUT/'
list_filenames = glob.glob(path+"*.pdf")
list_folder_names = glob.glob("*.pdf")
#print(list_folder_names)

#print(list_filenames)
for path_file in list_filenames:

    
    outputfile_name = path_file.split("/")[-1]
    output_folder_name = path_file.split("/")[-1]
    output_folder_name = output_folder_name.split(".")[0]+"/"
    #print(output_folder_name)

    output_data = output_path+output_folder_name+outputfile_name
    #print(output_data)

    if os.path.exists(output_path+output_folder_name):
        continue
    else:
        os.makedirs(output_path+output_folder_name)

    with open(path_file, "rb") as f:
        pdf = pdftotext.PDF(f)


        
        # Extracting metadata
        #print(pdf[0])
        page_1_text= pdf[0]
        page_1_text= page_1_text.split('\n')
        #print(page_1_text)

        def get_data(page_1_text):
            __dict = {}

            for line in page_1_text:
                if ':' not in line:
                    continue
                key, value = line.split(':')
                __dict[key.strip()] = value.strip()
            return __dict

        page_data = get_data(page_1_text)
        
        with open(output_data.replace('.pdf', '_metadata.csv'), 'w+') as output:
            for key in page_data.keys():
                output.write("%s,%s\n"%(key,page_data[key]))

          

        # Extractiong sample summary
        page_2_text = pdf[1]
        page_2_text = page_2_text.split('\n')

        def get_data(page_2_text):
            __dict = {}
    
            for line in page_2_text:
                if ':' not in line:
                    continue
                key, value = line.split(':')
                __dict[key.strip()] = value.strip()
            return __dict

        page_data = get_data(page_2_text)

        with open(output_data.replace('.pdf', '_sample_summary.csv'), 'w') as f:
            for key in page_data.keys():
                f.write("%s,%s\n"%(key,page_data[key]))

        

        page_3_text = pdf[2]
        page_3_text = page_3_text.split('\n')


        # extracting dst_table

        def get_data(page_3_text):
            __dict = {}
            for line in page_3_text:
                if ')' not in line:
                    continue
                key, value = line.split(')')
                __dict[key.strip()] = value.strip()
            return __dict

        page_data = get_data(page_3_text)
        
        with open(output_data.replace('.pdf', '_dst_table.csv'), 'w') as f:
            for key in page_data.keys():
                f.write("%s,%s\n"%(key,page_data[key]))

        
    
        # extracting clinical summary
        clinical_summary = open(path_file.replace('.pdf','.txt'),'wb')
        text = pdf[2].encode("utf8")

        #print(text)

        clinical_summary.write(text)
        clinical_summary.write(bytes((12,)))  # write page delimiter (form feed 0x0C)   # decimal to hexadecimal
        clinical_summary.close()

        
        text = open(path_file.replace('pdf','txt'), "rt")
        readtext =text.readlines()
        
        del readtext[0]
        del readtext[2::]

        #print(readtext)

        new_file = open(path_file.replace('.pdf','.txt'),"w+")

        for line in readtext:
            line = line.replace("-\n","")
            line = line.replace("\n",",").strip()
            line = line.replace("    ",",").strip()

            #print(line)            
            new_file.write(line)
                    
        new_file.close()
            

        dataframe = pd.read_csv(path_file.replace('.pdf','.txt'))
        #print(dataframe)
        dataframe.to_csv(output_data.replace('.pdf','_clinical_summary.csv'))

        


        # Extracting mutation table

        df = read_pdf(path_file, pages=5, stream=True)
        if df == []:
            sample_data = open(output_data.replace('.pdf','_mutation_table.csv'),'w')
            sample_data.write("No Mutations were Detected")
            sample_data.close()
            #print("No Mutations were Detected!")
        else:
            col=len(df[0].columns)
            x='SN'         # titles adding to x
            for i in range(col):
                x=x+','+(df[0].columns)[i]
            value=(len(df[0][(df[0].columns)[0]]))    # total rows present
            w=''                              # row value
            for i in range(col):
                w=w+str(i)
                for j in range(value):
                    w=w+','+str(numpy.array(df[0][((df[0].columns)[j])])[i])
                w=w+"\n"
            sample_data=open(output_data.replace('.pdf','_mutation_table.csv'),'w')
            sample_data.write(x+'\n'+w)
            sample_data.close()



        # using pandas to edit csv files 
        # page 1 csv file    # metadata

        df = pd.read_csv(output_data.replace('.pdf', '_metadata.csv'))
        df = df.iloc[:-1]
        df.T.to_csv(output_data.replace('.pdf', '_metadata.csv'),header=False)

        # page 2 data     # sample_summary

        df = pd.read_csv(output_data.replace('.pdf', '_sample_summary.csv'))
        df = df.iloc[:-1]
        df.T.to_csv(output_data.replace('.pdf', '_sample_summary.csv'),header=False)

        # page 3 csv     # dst_table

        df = pd.read_csv(output_data.replace('.pdf', '_dst_table.csv'))        
        df = df.iloc[:-1]
        df.T.to_csv(output_data.replace('.pdf', '_dst_table.csv'),header=False)

        df = pd.read_csv(output_data.replace('.pdf', '_dst_table.csv'))
        df = df.add_suffix(")")
        df.to_csv(output_data.replace('.pdf', '_dst_table.csv'),index=None)


        # page 4 csv           # mutation_table

        df = pd.read_csv(output_data.replace('.pdf','_mutation_table.csv'))
        #print(df.columns[0])
        if df.columns[0] != "SN":
            continue
        else:
            df.pop('SN')
            df = df.fillna("")
            df.to_csv(output_data.replace('.pdf','_mutation_table.csv'), index=False)


        
        # using pandas to edit csv file
        # page 3 csv    # clinical summary
        
        dataframe = pd.read_csv(output_data.replace('.pdf','_clinical_summary.csv'))
        print(dataframe)
        dataframe = dataframe.loc[:,~dataframe.columns.str.match("Unnamed")]
        dataframe.T.to_csv(output_data.replace('.pdf','_clinical_summary.csv'),header=False)

        dataframe = pd.read_csv(output_data.replace('.pdf','_clinical_summary.csv'))

        x=dataframe.columns
        a=(str(dataframe.columns).replace("Index(['", "")).replace("'], dtype='object')", "")

        col=a
        row=dataframe[a][0]

        for i in range(int(len(dataframe)/2)):
            col=col+','+dataframe[a][2*i+1]
            row=row+','+dataframe[a][2*i+2]
    
        new_csv=open(output_data.replace('.pdf','_clinical_summary.csv'),'w')
        new_csv.write(col+'\n'+row)
        new_csv.close()




In [5]:
# reading all the csv files form the output folder

output_path = "/home/aman/Desktop/HaystackAnalytics/Task1/genomeanalysisfinalreport/OUTPUT/"

list_folder_names = glob.glob(output_path+"/*")
#print(list_folder_names)
list_file_names = glob.glob(output_path+"/*"+"/*.csv")
#print(list_file_names)



for csv_files in list_file_names:
    #print(csv_files)
    if '_mutation_table.csv' in csv_files:
        continue
    df = pd.read_csv(csv_files)
    #print(df)
    #df.to_csv('file_name.csv')

    files = os.path.join(csv_files,"combinedsummary.csv")
    files = glob.glob(files)

    '''
    files = glob.glob(df)
    # joining files with concat and read_csv
    df1 = pd.concat(map(pd.read_csv, files), ignore_index=True)
    print(df1)
    '''
#files = os.path.join("C:\\Users\\amit_\\Desktop\\", "sales*.csv")


In [ ]:
# setting the path for joining multiple files
files = os.path.join("C:\\Users\\amit_\\Desktop\\", "sales*.csv")

# list of merged files returned
files = glob.glob(files)

print("Resultant CSV after joining all CSV files at a particular location...");

# joining files with concat and read_csv
df = pd.concat(map(pd.read_csv, files), ignore_index=True)
print(df)